# Predicting Income with Social Data
## Practice Skeleton — Linear Regression in R (PSID 2017)

**Goal:** Practice the full linear-regression workflow in R (assumptions → cleaning → train/test split → simple & multiple LM → fit assessment → coefficient interpretation) while predicting labor income from social and demographic variables in the Panel Study of Income Dynamics (PSID).

**Context:** You are a data analyst supporting a social-policy research team. The team wants to understand how education, age, and gender relate to labor-derived income so they can brief both technical researchers and non-technical policy audiences.

**Data:** `data/psid_2017.csv` — a cross-section of PSID respondents with variables:
- `gender` (1 = Male, 2 = Female)
- `age`
- `married` (0/1/2/3 codes)
- `employed` (character status)
- `educated_in_us`
- `highest_degree`
- `education_years`
- `labor_income` (yearly labor earnings)

**Flowchart of the desired outcome:**

![Predicting Income Pipeline](predicting_income_flowchart.png)

---
### How to use this notebook
- Complete each step in the code cells (look for `# YOUR CODE HERE`).
- Run cells in order (requires an R kernel).
- Compare with the companion **Solution** notebook when stuck.
- At the end: alternate solutions, more practice drills, and a tunable simulation.
- A detailed **Cheat Sheet** is included at the bottom of this notebook (and the Solution notebook).

---
### Audience note (from the attached PDFs)
Remember the four classic audience types (Experts / Technicians / Executives / Nonspecialists) and the checklist dimensions (data literacy, subject knowledge, needs & interests, culture). Your final narrative should be adaptable: technical detail for the research supervisor, clear headlines + policy implication for the executive, and plain-language explanation for a nonspecialist briefing.

## 0. Setup — Load packages and data

In [ ]:
# Load required packages
library(dplyr)
library(ggplot2)
# modelr is optional; we can use base predict() if it is not installed

# Load the PSID data
psid <- read.csv("data/psid_2017.csv")

# Quick peek
head(psid)

## 1. Clean and check data assumptions

### Task 1 — Inspect structure
Call `str()` on `psid` and confirm the variables listed in the introduction are present. Note any unexpected types or codes.

In [ ]:
# YOUR CODE HERE
str(psid)

### Task 2 — Age distribution (raw)
Create a bar or histogram chart of `age`. Do any observed values look unrealistic (e.g., age = 999)?

In [ ]:
# YOUR CODE HERE — ggplot + geom_bar or geom_histogram
age_raw_plot <- psid %>%
  ggplot(aes(age)) +
  geom_histogram(binwidth = 5, fill = "#1f77b4", color = "white") +
  labs(title = "Age Distribution (Raw)", x = "Age", y = "Count") +
  theme_minimal()
age_raw_plot

### Task 3 — Filter to working age
Filter to respondents roughly of working age (18 ≤ age ≤ 75). Save the result as `psid_age`.

In [ ]:
# YOUR CODE HERE
psid_age <- psid %>%
  filter(age >= 18, age <= 75)
nrow(psid_age)

### Task 4 — Confirm age filter
Re-plot the age distribution on the filtered data.

In [ ]:
# YOUR CODE HERE
age_clean_plot <- psid_age %>%
  ggplot(aes(age)) +
  geom_histogram(binwidth = 5, fill = "#2ca02c", color = "white") +
  labs(title = "Age Distribution (18-75)", x = "Age", y = "Count") +
  theme_minimal()
age_clean_plot

### Task 5 — Education years boxplot
Create a boxplot of `education_years`. Look for extreme values (e.g., 99).

In [ ]:
# YOUR CODE HERE
educ_box <- psid_age %>%
  ggplot(aes(x = "", y = education_years)) +
  geom_boxplot(fill = "#ff7f0e") +
  labs(title = "Education Years", x = "", y = "Years of formal education") +
  theme_minimal()
educ_box

### Task 6 — Filter education years
Keep only observations where `education_years` is between 5 and 25 inclusive. Overwrite or create `psid_clean`.

In [ ]:
# YOUR CODE HERE
psid_clean <- psid_age %>%
  filter(education_years >= 5, education_years <= 25)
nrow(psid_clean)

### Task 7 — Labor income boxplot
Plot a boxplot of `labor_income` on the current cleaned data.

In [ ]:
# YOUR CODE HERE
income_box <- psid_clean %>%
  ggplot(aes(x = "", y = labor_income)) +
  geom_boxplot(fill = "#d62728") +
  labs(title = "Labor Income", x = "", y = "Yearly labor income ($)") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)
income_box

### Task 8 — Summary of labor_income
Call `summary()` on `labor_income`. What does the high proportion of zeros tell you?

In [ ]:
# YOUR CODE HERE
summary(psid_clean$labor_income)
cat("Share with positive income:", mean(psid_clean$labor_income > 0), "\n")

### Task 9 — Mean income by age (and decide on further filter)
Using `group_by` + `summarise`, compute mean labor_income by age and plot with `geom_point`.  
**Decision point:** Many zeros dilute the linear relationship. For the rest of the project we will also filter to `labor_income > 0` so that we model the income of earners. (You can experiment later without this filter.)

In [ ]:
# YOUR CODE HERE — mean income by age scatter
mean_by_age <- psid_clean %>%
  group_by(age) %>%
  summarise(mean_income = mean(labor_income, na.rm = TRUE), .groups = "drop")

ggplot(mean_by_age, aes(age, mean_income)) +
  geom_point(color = "#1f77b4") +
  labs(title = "Mean Labor Income by Age", x = "Age", y = "Mean income") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)

# Further filter to positive earners (recommended for clearer teaching results)
psid_clean <- psid_clean %>% filter(labor_income > 0)
cat("Final analytic n:", nrow(psid_clean), "\n")

## 2. Build model and assess fit

### Task 10 — Train / test split (60/40)
Set a seed, create a logical sample vector, and build `train` and `test` data frames from `psid_clean`.

In [ ]:
set.seed(123)
# YOUR CODE HERE
sample_idx <- sample(c(TRUE, FALSE), nrow(psid_clean), replace = TRUE, prob = c(0.6, 0.4))
train <- psid_clean[sample_idx, ]
test  <- psid_clean[!sample_idx, ]
cat("Train n:", nrow(train), "  Test n:", nrow(test), "\n")

### Task 11 — Simple linear model
Fit `labor_income ~ education_years` on the training data. Store the result as `model`.

In [ ]:
# YOUR CODE HERE
model <- lm(labor_income ~ education_years, data = train)
summary(model)

### Task 12 — Visual check vs LOESS
Scatter the training points, overlay the OLS line (`method = "lm"`) and a LOESS smoother (different color, no SE).

In [ ]:
# YOUR CODE HERE
ggplot(train, aes(education_years, labor_income)) +
  geom_point(alpha = 0.3, color = "#1f77b4") +
  geom_smooth(method = "lm", se = TRUE, color = "red") +
  geom_smooth(method = "loess", se = FALSE, color = "darkgreen", linetype = "dashed") +
  labs(title = "Income vs Education (Train)",
       subtitle = "Red = OLS, Green dashed = LOESS",
       x = "Education years", y = "Labor income ($)") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)

### Task 13 — Extract R-squared
Extract `r.squared` from the model summary, multiply by 100, and store as `r_sq`.

In [ ]:
# YOUR CODE HERE
r_sq <- summary(model)$r.squared * 100
r_sq

### Task 14 — Narrative for simple model
Uncomment / adapt the sprintf (or write your own sentence) that reports the R² of the simple model.

In [ ]:
# YOUR CODE HERE (uncomment and fill)
# sprintf("Based on a simple linear regression, approximately %.1f percent of the variation in labor income can be explained by years of formal education alone.", r_sq)

## 3. Build comparison model and analyze results

### Task 15 — Multiple linear model
Fit `labor_income ~ education_years + age + gender`.  
First convert `gender` to a factor with meaningful labels if you have not already done so.

In [ ]:
# YOUR CODE HERE — ensure gender is a factor
train <- train %>% mutate(gender = factor(gender, levels = c(1, 2), labels = c("Male", "Female")))
test  <- test  %>% mutate(gender = factor(gender, levels = c(1, 2), labels = c("Male", "Female")))

model_2 <- lm(labor_income ~ education_years + age + gender, data = train)
summary(model_2)

### Task 16 — R-squared of model_2

In [ ]:
# YOUR CODE HERE
r_sq_2 <- summary(model_2)$r.squared * 100
r_sq_2

### Task 17 — Narrative for multiple model

In [ ]:
# YOUR CODE HERE
# sprintf("Adding age and gender raises the explained variation to approximately %.1f percent.", r_sq_2)

### Task 18 — Observed vs predicted (test set)
Add predictions from `model_2` to the test set and plot observed income vs age with the predicted line overlaid.

In [ ]:
# YOUR CODE HERE
test$pred <- predict(model_2, newdata = test)

ggplot(test, aes(age, labor_income)) +
  geom_point(alpha = 0.25, color = "gray40") +
  geom_line(aes(y = pred), color = "blue", linewidth = 0.9) +
  labs(title = "Observed vs Predicted (Model 2 on Test)",
       x = "Age", y = "Labor income ($)") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)

### Task 19 — Interpret coefficients
Look at `summary(model_2)`:
- Are the three predictors statistically significant?
- How do you interpret the coefficient on the factor `genderFemale`?
- Which continuous predictor has the larger absolute effect (per unit)?

In [ ]:
# YOUR CODE HERE — just print the summary again if needed
summary(model_2)$coefficients

### Task 20 — Extract education coefficient

In [ ]:
# YOUR CODE HERE
education_coefficient <- coef(model_2)["education_years"]
education_coefficient

### Task 21 — Narrative for the education coefficient (ceteris paribus)

In [ ]:
# YOUR CODE HERE
# sprintf("Holding age and gender constant, each additional year of formal education is associated with an estimated $%.0f increase in annual labor income.", education_coefficient)

### Task 22 — Reflection
You have cleaned a real-world social survey, built and compared simple vs multiple linear models, quantified fit, and produced audience-ready coefficient narratives.  
Feel free to experiment with additional predictors (`married`, `highest_degree`, etc.) in the More Practice section.

---
## Alternate Code Paths

The same results can be obtained with slightly different R idioms.

In [ ]:
# Alternate 1: base-R subsetting instead of dplyr filter
# psid_clean_alt <- psid[psid$age >= 18 & psid$age <= 75 &
#                        psid$education_years >= 5 & psid$education_years <= 25 &
#                        psid$labor_income > 0, ]

# Alternate 2: formula interface with I() or poly for non-linear age (optional)
# model_poly <- lm(labor_income ~ education_years + poly(age, 2) + gender, data = train)

# Alternate 3: extract R² with broom (if installed) or directly
# r_sq_alt <- summary(model)$adj.r.squared   # adjusted version

# Alternate 4: predictions with base predict() (already used) vs modelr::add_predictions
# if (requireNamespace("modelr", quietly = TRUE)) {
#   test <- modelr::add_predictions(test, model_2)
# }

---
## More Practice

1. Add `married` (re-coded as a factor) to the multiple model. Does R² improve meaningfully?
2. Create a dummy for “college or higher” from `highest_degree` and compare its coefficient with the continuous `education_years`.
3. Examine residuals of `model_2` (histogram + residual-vs-fitted). Are the classical OLS assumptions (linearity, homoscedasticity, normality of residuals) reasonable?
4. Compute the percentage error (RSE / mean(y)) for both models on the test set.
5. Write two short paragraphs: one for a technical supervisor and one for a non-specialist policy briefing that explain the education coefficient.

In [ ]:
# Space for your more-practice experiments
# YOUR CODE HERE

---
## Simulation Section — Sensitivity to sample size & noise

Modify the parameters below and re-run to see how R² and the education coefficient change.

In [ ]:
# === TUNABLE PARAMETERS ===
n_sim          <- 30          # number of Monte-Carlo replications
sample_frac    <- 0.6         # fraction of clean data used as "train" each time
noise_sd_mult  <- 1.0         # multiply residual SD by this factor (1 = original)
min_edu        <- 5           # education_years lower bound
max_edu        <- 25          # education_years upper bound
age_lo         <- 18
age_hi         <- 75
# ============================

set.seed(42)
sim_results <- replicate(n_sim, {
  # re-filter with possibly different bounds
  d <- psid %>%
    filter(age >= age_lo, age <= age_hi,
           education_years >= min_edu, education_years <= max_edu,
           labor_income > 0) %>%
    mutate(gender = factor(gender, levels = c(1,2), labels = c("Male","Female")))
  idx <- sample(c(TRUE, FALSE), nrow(d), replace = TRUE, prob = c(sample_frac, 1-sample_frac))
  tr <- d[idx, ]
  # optional extra noise
  if (noise_sd_mult != 1) {
    tr$labor_income <- tr$labor_income + rnorm(nrow(tr), 0, sd(tr$labor_income, na.rm=TRUE)*(noise_sd_mult-1))
  }
  m <- lm(labor_income ~ education_years + age + gender, data = tr)
  c(r2 = summary(m)$r.squared,
    edu_coef = coef(m)["education_years"])
})

sim_df <- as.data.frame(t(sim_results))
cat("Mean R² across simulations:", mean(sim_df$r2), "\n")
cat("Mean education coefficient:", mean(sim_df$edu_coef), "\n")

par(mfrow = c(1,2))
hist(sim_df$r2, main = "Distribution of R²", xlab = "R²", col = "skyblue")
hist(sim_df$edu_coef, main = "Distribution of Education Coef", xlab = "$", col = "salmon")
par(mfrow = c(1,1))

---
## Cheat Sheet — Linear Regression in R (PSID / Social Data)

### Data inspection & cleaning
```r
str(df)                          # structure
summary(df$var)                  # five-number + mean
table(df$cat, useNA = "ifany")   # frequency incl. NA
df %>% filter(cond1, cond2)      # dplyr filter
df[df$age >= 18 & df$age <= 75, ]# base-R equivalent
```

### Visualization (ggplot2)
```r
ggplot(df, aes(x)) + geom_histogram(binwidth = 5)
ggplot(df, aes(x = "", y = var)) + geom_boxplot()
ggplot(df, aes(x, y)) + geom_point(alpha = 0.3) +
  geom_smooth(method = "lm") + geom_smooth(method = "loess", se = FALSE)
```

### Train / test split
```r
set.seed(123)
idx <- sample(c(TRUE, FALSE), nrow(df), replace = TRUE, prob = c(0.6, 0.4))
train <- df[idx, ];  test <- df[!idx, ]
```

### Modeling
```r
model  <- lm(y ~ x, data = train)                 # simple
model2 <- lm(y ~ x1 + x2 + factor(x3), data = train)  # multiple
summary(model)                                   # coefficients, R², RSE, p-values
summary(model)$r.squared
coef(model)["x1"]
sigma(model)                                     # residual SE
predict(model, newdata = test)                   # predictions
```

### Fit metrics & interpretation
- **R²** = proportion of variance in y explained by the model (0–1).
- **RSE / mean(y)** ≈ average percentage error.
- Continuous coefficient: “holding other variables constant, a 1-unit increase in X is associated with a β change in Y.”
- Binary factor coefficient: “the difference in expected Y between the level and the reference level.”

### Common pitfalls with survey data
- Many zeros → consider filtering to positive earners or using a two-part / log model.
- Coded missing (99, 999, 9) → always inspect ranges and `table()`.
- Factor vs numeric: convert categorical predictors with `factor()` and meaningful labels.

### Audience adaptation (quick checklist)
| Audience        | Depth              | Language                     | Visuals              |
|-----------------|--------------------|------------------------------|----------------------|
| Expert / Tech   | Full coefficients, SE, p, residual diagnostics | Statistical terms OK | Residual plots, R² comparison |
| Executive       | Headline R², key β, policy implication | Minimal jargon            | One clear scatter + line |
| Nonspecialist   | “Each extra year of school ≈ $X more income” | Everyday words            | Simple bar or annotated scatter |

---
*End of Practice Skeleton. See the Solution notebook for complete filled cells and additional commentary.*